In [3]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# Week 4: Baseline Linear Regression Model

## Objective

The objective of this notebook is to build and evaluate a baseline Linear Regression model for predicting California residential home sale prices.

The cleaned dataset produced during Week 3 is used as the starting point. Since the data has already undergone filtering, feature selection, missing value handling, and categorical encoding, this notebook focuses entirely on model development and evaluation.

The resulting performance metrics will establish a baseline benchmark that can be compared against more advanced machine learning models in future weeks.

In [4]:
df = pd.read_csv("cleaned_california_sales.csv")

print(df.shape)
df.head()

(385497, 459)


,CloseDate,Latitude,Longitude,LivingArea,DaysOnMarket,ListingKeyNumeric,ParkingTotal,LotSizeAcres,YearBuilt,StreetNumberNumeric,...,"Levels_Two,ThreeOrMore","Levels_Two,ThreeOrMore,MultiSplit",Levels_Unknown,NewConstructionYN_True,NewConstructionYN_Unknown,latfilled_True,latfilled_Unknown,lonfilled_True,lonfilled_Unknown,ClosePrice
0,2023-07-03,33.974378,-118.295831,1076.0,228.0,554248930.0,4.0,0.1375,1920.0,1156.0,...,False,False,False,False,False,False,False,False,False,500000.0
1,2023-10-27,34.491896,-117.374508,2669.0,5.0,552093317.0,3.0,0.1700,1991.0,13231.0,...,False,False,True,False,True,False,False,False,False,331000.0
2,2023-07-05,33.782033,-116.993451,1644.0,172.0,552062457.0,2.0,0.1700,2006.0,547.0,...,False,False,False,False,False,False,False,False,False,410000.0
3,2023-07-19,33.895115,-118.195298,918.0,7.0,551932712.0,2.0,0.1094,1950.0,15202.0,...,False,False,False,False,False,False,False,False,False,519000.0
4,2023-06-26,34.135025,-117.809712,5110.0,97.0,544556015.0,3.0,0.5100,2023.0,1618.0,...,False,False,False,True,False,False,False,False,False,2999822.9


In [7]:
print(f"Dataset Shape: {df.shape}")

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

df = df.dropna(subset=["Latitude", "Longitude"])

print(f"Dataset Shape After Removing Missing Coordinates: {df.shape}")

print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())


Dataset Shape: (385497, 459)

Column Names:
['CloseDate', 'Latitude', 'Longitude', 'LivingArea', 'DaysOnMarket', 'ListingKeyNumeric', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt', 'StreetNumberNumeric', 'BathroomsTotalInteger', 'BuyerAgencyCompensation', 'BuildingAreaTotal', 'BedroomsTotal', 'BelowGradeFinishedArea', 'Stories', 'LotSizeArea', 'MainLevelBedrooms', 'GarageSpaces', 'AssociationFee', 'LotSizeSquareFeet', 'Flooring_Bamboo,Brick,Carpet', 'Flooring_Bamboo,Brick,Carpet,Stone', 'Flooring_Bamboo,Brick,Tile', 'Flooring_Bamboo,Brick,Wood', 'Flooring_Bamboo,Carpet', 'Flooring_Bamboo,Carpet,Concrete', 'Flooring_Bamboo,Carpet,Concrete,Laminate,SeeRemarks,Stone', 'Flooring_Bamboo,Carpet,Concrete,SeeRemarks,Tile,Wood', 'Flooring_Bamboo,Carpet,Concrete,Tile', 'Flooring_Bamboo,Carpet,Concrete,Wood', 'Flooring_Bamboo,Carpet,Laminate', 'Flooring_Bamboo,Carpet,Laminate,SeeRemarks,Tile', 'Flooring_Bamboo,Carpet,Laminate,SeeRemarks,Tile,Wood', 'Flooring_Bamboo,Carpet,Laminate,Stone', 'Flooring

## Convert CloseDate to Datetime


The `CloseDate` column is currently stored as a text (`object`) data type. To perform a temporal train/test split, the dates must first be converted into a datetime format.

Using a chronological split better reflects how a predictive model would be used in practice. The model is trained on historical home sales and evaluated on the most recent month of data, preventing future information from influencing the training process.

In [8]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"])

print(df["CloseDate"].dtype)

print("\nDate Range:")
print(df["CloseDate"].min())
print(df["CloseDate"].max())

datetime64[ns]

Date Range:
2023-06-01 00:00:00
2026-05-31 00:00:00


## Temporal Train/Test Split

To simulate real-world home price prediction, the most recent full month of sales is reserved as the test set. This ensures the model is evaluated on future data that was not seen during training.

Following our team's modeling strategy, the training set includes **all available sales from June 2023 through the month immediately preceding the test month**. June 2023 was selected because earlier months contained substantially fewer transactions and were considered less representative of current market conditions.

Using a chronological split helps prevent data leakage and provides a more realistic evaluation of how the model would perform when predicting future home sale prices.

In [9]:
# Create monthly periods from CloseDate
df["YearMonth"] = df["CloseDate"].dt.to_period("M")

# Training starts in June 2023
training_start = pd.Period("2023-06", freq="M")

# Most recent month is the test month
latest_month = df["YearMonth"].max()

# Chronological split
train_df = df[
    (df["YearMonth"] >= training_start) &
    (df["YearMonth"] < latest_month)
].copy()

test_df = df[
    df["YearMonth"] == latest_month
].copy()

print(f"Training Period: {train_df['YearMonth'].min()} to {train_df['YearMonth'].max()}")
print(f"Test Month: {latest_month}")

print(f"\nTraining Set Shape: {train_df.shape}")
print(f"Testing Set Shape: {test_df.shape}")

Training Period: 2023-06 to 2026-04
Test Month: 2026-05

Training Set Shape: (373520, 460)
Testing Set Shape: (11823, 460)


In [12]:
training_months = sorted(train_df["YearMonth"].unique())

print("First 3 Training Months:")
print(training_months[:3])

print("\nLast 3 Training Months:")
print(training_months[-3:])

print("\nTest Month:")
print(test_df["YearMonth"].unique())

print(f"\nTraining Records: {len(train_df):,}")
print(f"Testing Records: {len(test_df):,}")

First 3 Training Months:
[Period('2023-06', 'M'), Period('2023-07', 'M'), Period('2023-08', 'M')]

Last 3 Training Months:
[Period('2026-02', 'M'), Period('2026-03', 'M'), Period('2026-04', 'M')]

Test Month:
<PeriodArray>
['2026-05']
Length: 1, dtype: period[M]

Training Records: 373,520
Testing Records: 11,823


## Separate Features and Target

The objective of the baseline model is to predict the home's sale price (`ClosePrice`), making it the target variable (`y`).

The `CloseDate` and `YearMonth` columns are excluded from the feature set because they were used solely to create the chronological train/test split. Removing these columns ensures the model does not directly use temporal information during training while still being evaluated on future sales.

The remaining variables serve as the predictor features (`X`) used to train the Linear Regression model.

In [13]:
# Create training features and target
X_train = train_df.drop(columns=["ClosePrice", "CloseDate", "YearMonth"])
y_train = train_df["ClosePrice"]

# Create testing features and target
X_test = test_df.drop(columns=["ClosePrice", "CloseDate", "YearMonth"])
y_test = test_df["ClosePrice"]

print(f"Training Features Shape: {X_train.shape}")
print(f"Training Target Shape: {y_train.shape}")

print(f"\nTesting Features Shape: {X_test.shape}")
print(f"Testing Target Shape: {y_test.shape}")

Training Features Shape: (373520, 457)
Training Target Shape: (373520,)

Testing Features Shape: (11823, 457)
Testing Target Shape: (11823,)


## Convert Boolean Features to Numeric

Many of the categorical variables were one-hot encoded during preprocessing, resulting in boolean (`True`/`False`) features.

Although scikit-learn can internally interpret boolean values, converting them to integers (`1` and `0`) creates a fully numeric feature matrix and ensures consistency throughout the modeling pipeline.

In [15]:
# Convert boolean columns to integers
bool_columns = X_train.select_dtypes(include="bool").columns

X_train[bool_columns] = X_train[bool_columns].astype(int)
X_test[bool_columns] = X_test[bool_columns].astype(int)

print(X_train.dtypes.value_counts())

int64      437
float64     20
Name: count, dtype: int64


## Verify Feature Data Types

Scikit-learn's Linear Regression model requires all predictor variables to be numeric. Before training the model, the feature matrix is inspected to verify that no unsupported data types remain after preprocessing.

This verification helps identify any remaining object or datetime columns that could prevent the model from fitting successfully.

In [14]:
print(X_train.dtypes.value_counts())

bool       437
float64     20
Name: count, dtype: int64


## Train Baseline Linear Regression Model


Linear Regression serves as the baseline model for this project because it is simple, interpretable, and computationally efficient.

Although more sophisticated machine learning algorithms may ultimately provide better predictive performance, establishing a baseline allows future models to be compared against a common reference point. This benchmark will help determine whether more complex models provide meaningful improvements in predictive accuracy.

In [16]:
model = LinearRegression()

model.fit(X_train, y_train)

print("Baseline Linear Regression model trained successfully.")

Baseline Linear Regression model trained successfully.


## Generate Predictions

Once the Linear Regression model has been trained, it is used to predict home sale prices for the unseen test dataset.

These predictions are then compared with the actual sale prices to evaluate how well the baseline model generalizes to future market conditions.

In [17]:
y_pred = model.predict(X_test)

print(f"Generated {len(y_pred):,} predictions.")

Generated 11,823 predictions.


## Evaluate Model Performance

The baseline model is evaluated using three commonly used regression metrics:

- **R² (Coefficient of Determination):** Measures how much of the variation in home sale prices is explained by the model.
- **Mean Absolute Error (MAE):** Measures the average prediction error in dollars.
- **Root Mean Squared Error (RMSE):** Measures the typical prediction error while penalizing larger errors more heavily.

Together, these metrics provide a comprehensive assessment of the baseline model's predictive performance.

In [19]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

results = pd.DataFrame({
    "Metric": ["R² Score", "Mean Absolute Error", "Root Mean Squared Error"],
    "Value": [r2, mae, rmse]
})

print("Baseline Model Performance")
print("-" * 35)

print(f"R² Score : {r2:.4f}")
print(f"MAE       : ${mae:,.2f}")
print(f"RMSE      : ${rmse:,.2f}")

results

Baseline Model Performance
-----------------------------------
R² Score : 0.6423
MAE       : $329,360.86
RMSE      : $495,452.73


,Metric,Value
0,R² Score,0.642306
1,Mean Absolute Error,329360.863776
2,Root Mean Squared Error,495452.726213


# Baseline Model Summary and Key Findings

## Features Used in the Model

The baseline Linear Regression model was trained using **457 predictor variables** generated from the cleaned CRMLS residential sales dataset.

The model incorporated several categories of information, including:

- **Property characteristics**
  - Living Area
  - Lot Size
  - Year Built
  - Number of Parking Spaces
  - Days on Market
  - Geographic Coordinates (Latitude and Longitude)

- **Location information**
  - Geographic coordinates
  - Remaining location-based indicators retained after preprocessing

- **Property attributes**
  - Features describing garages, pools, fireplaces, waterfront status, view status, flooring type, construction details, and other physical property characteristics.

- **Categorical variables**
  - Categorical features that remained after preprocessing were converted into one-hot encoded binary variables. These account for the majority of the predictor variables used by the model.

During preprocessing, several variables were intentionally removed because they were either identifiers, introduced target leakage, or contained extremely high-cardinality values that would not improve a baseline Linear Regression model.

Examples include:

- Listing Price
- Original Listing Price
- Listing IDs
- Agent names
- Office names
- Addresses
- School names
- Builder names
- City names
- Other high-cardinality categorical variables

This left a final feature matrix containing **457 numeric predictor variables** suitable for machine learning.

---

# Baseline Model Performance

The Linear Regression model produced the following results on the held-out **May 2026** test dataset:

| Metric | Result |
|--------|--------:|
| **R² Score** | **0.6423** |
| **Mean Absolute Error (MAE)** | **\$329,360.86** |
| **Root Mean Squared Error (RMSE)** | **\$495,452.73** |

---

# Interpretation of Results

The baseline Linear Regression model explained approximately **64.2% of the variation** in California residential home sale prices for the unseen May 2026 dataset.

The **Mean Absolute Error (MAE)** indicates that, on average, predicted sale prices differed from the true sale prices by approximately **\$329,361**.

The **Root Mean Squared Error (RMSE)** was approximately **\$495,453**, which is noticeably larger than the MAE. This suggests that while many predictions were reasonably accurate, the model produced larger prediction errors for certain homes. These larger errors are expected in a statewide housing dataset because luxury properties, unique homes, and markets with rapidly changing prices are generally more difficult to predict.

---

# Key Takeaways

Several important conclusions can be drawn from the baseline model:

- The preprocessing pipeline successfully produced a clean, fully numeric dataset that could be used directly for machine learning.

- Using historical sales from **June 2023 through April 2026** and evaluating on **May 2026** created a realistic chronological train/test split that avoids data leakage.

- The baseline Linear Regression model captured many of the overall pricing trends across California, explaining roughly two-thirds of the variation in home sale prices.

- Despite this, prediction errors remain relatively large, indicating that a simple linear model cannot fully represent the complex relationships present in residential real estate data.

- Housing prices are influenced by nonlinear interactions among property characteristics, location, neighborhood effects, and market conditions that Linear Regression may not capture effectively.

- These baseline results establish a benchmark for future machine learning models. More advanced algorithms such as Random Forest, Gradient Boosting, or XGBoost can now be evaluated against this baseline to determine whether they provide meaningful improvements in predictive performance.

---

# Conclusion

This notebook established a complete baseline machine learning workflow for predicting California residential home sale prices. Beginning with the cleaned dataset from Week 3, the data was chronologically split into training and testing sets, a Linear Regression model was trained, and its predictive performance was evaluated using standard regression metrics.

The baseline model provides a strong reference point for future experimentation. While it captures many general pricing trends, the results also demonstrate that additional modeling techniques will likely be necessary to better capture the complexity of the California housing market and improve prediction accuracy.